In [1]:
import glob
import os
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

train = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv")
submission = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv")


/tmp/ipykernel_16/2008091755.py:11: DtypeWarning: Columns (38,39) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv")


In [2]:
import glob
import os
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error

In [3]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
submission = pd.read_csv("sample_submission.csv")

TARGET = "TargetValue"
ID_COLUMN = "TransactionID"

test_ids = test[ID_COLUMN].copy()

all_features = pd.concat([train.drop(columns=[TARGET]), test], ignore_index=True)

date = pd.to_datetime(all_features["TransactionDate"], errors="coerce")

all_features["TransactionYear"] = date.dt.year
all_features["TransactionMonth"] = date.dt.month

all_features["ManufactureYear"] = all_features["ManufactureYear"].where(
    all_features["ManufactureYear"].between(1900, 2020)
)

all_features["AssetAge"] = all_features["TransactionYear"] - all_features["ManufactureYear"]
all_features["AssetAge"] = all_features["AssetAge"].where(all_features["AssetAge"].between(0, 100))

all_features["LogOperationalHours"] = np.log1p(
    all_features["OperationalHoursMeter"].clip(lower=0)
)

for col in ["ProductConfigID", "Spec_FullDescriptor"]:
    all_features[f"{col}_Frequency"] = all_features[col].map(
        all_features[col].value_counts(dropna=False)
    ).astype("float32")

all_features = all_features.drop(
    columns=["AssetID", ID_COLUMN, "TransactionDate"],
    errors="ignore"
)

categorical_columns = all_features.select_dtypes(include="object").columns.tolist()

if "ProductConfigID" in all_features.columns:
    categorical_columns.append("ProductConfigID")

categorical_columns = list(dict.fromkeys(categorical_columns))

for col in categorical_columns:
    all_features[col] = all_features[col].fillna("__MISSING__").astype("category")

X = all_features.iloc[:len(train)].copy()
X_test = all_features.iloc[len(train):].copy()
y = np.log1p(train[TARGET])

bins = pd.qcut(y, q=20, labels=False, duplicates="drop")

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=bins
)

params = {
    "objective": "regression",
    "metric": "rmse",
    "n_estimators": 7000,
    "learning_rate": 0.03,
    "num_leaves": 63,
    "min_child_samples": 40,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "cat_smooth": 20,
    "cat_l2": 10,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
    "force_col_wise": True
}

model = lgb.LGBMRegressor(**params)

model.fit(
    X_train,
    y_train,
    categorical_feature=categorical_columns,
    eval_set=[(X_valid, y_valid)],
    eval_metric="rmse",
    callbacks=[
        lgb.early_stopping(150, verbose=True),
        lgb.log_evaluation(100)
    ]
)

valid_pred = np.clip(
    np.expm1(model.predict(X_valid, num_iteration=model.best_iteration_)),
    0,
    None
)

rmsle = np.sqrt(mean_squared_log_error(np.expm1(y_valid), valid_pred))

print("Best iteration:", model.best_iteration_)
print("Local validation RMSLE:", rmsle)

params["n_estimators"] = max(100, int(model.best_iteration_ * 1.05))

final_model = lgb.LGBMRegressor(**params)
final_model.fit(X, y, categorical_feature=categorical_columns)

submission[ID_COLUMN] = test_ids
submission[TARGET] = np.clip(np.expm1(final_model.predict(X_test)), 0, None)

submission.to_csv("submission.csv", index=False)

print("Created submission.csv")
print(submission.head())

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

# MILESTONE-1

In [ ]:
print(train.shape,test.shape)

In [ ]:
set(train.columns)-set(test.columns)

In [ ]:
cols=train.select_dtypes(include='number').columns
len(cols)


In [ ]:
missing=train.isna().sum()
missing[missing > 124800]

In [ ]:
train['OperationalHoursMeter'].isna().mean()*100

In [ ]:
train['TargetValue'].median()

In [ ]:
train['ManufactureYear'].mode()[0]

In [ ]:
train['TransactionDate']=pd.to_datetime(train['TransactionDate'])
train['TransactionDate'].min()

In [ ]:
train['Spec_BaseClass'].nunique()

In [ ]:
train['RegionCode'].value_counts().head()

In [ ]:
train[train['RegionCode']=='Florida']['TargetValue'].mean()

In [ ]:
train['UtilizationTier'].value_counts(dropna=False)

In [ ]:
train[['TargetValue','OperationalHoursMeter']].corr()

In [ ]:
train['FunctionalClassification'].value_counts().head()